# RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval

**A deep-dive, from-scratch implementation — no LangChain, no LlamaIndex, no OpenAI API.**

| Component | Choice |
|---|---|
| LLM | `Qwen/Qwen3-14B` (4-bit NF4 via bitsandbytes) |
| Embeddings | `BAAI/bge-base-en-v1.5` (sentence-transformers) |
| Vector Store | FAISS (flat L2 index) |
| Clustering | scikit-learn KMeans / Gaussian Mixture |
| Framework | Pure Python — every helper written inline |

> **Paper**: *RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval*  
> Sarthi et al., ICLR 2024 — [arXiv 2401.18059](https://arxiv.org/abs/2401.18059)

<div style="text-align: center;">

<img src="../images/raptor.svg" alt="RAPTOR" style="width:100%; height:auto;">
</div>

## 1 · The Granularity Problem

Every RAG system faces a fundamental tension:

| Strategy | Strengths | Weaknesses |
|---|---|---|
| **Small chunks** (200-400 tokens) | Great for pinpoint factual recall | Loses cross-paragraph themes, causes fragmented retrieval |
| **Large chunks** (2000+ tokens) | Preserves narrative flow | Dilutes relevance for specific questions, wastes context window |
| **Full-document embeddings** | Captures global themes perfectly | Terrible for detail questions; embedding compresses too much |

### The Core Dilemma

Consider a 50-page technical report on climate change:

- **Detail query**: *"What is the specific CO₂ concentration threshold for irreversible ice-sheet collapse?"*  
  → Needs a precise paragraph buried on page 37.
- **Thematic query**: *"What are the main arguments for why climate adaptation is more urgent than mitigation?"*  
  → Needs synthesis across Chapters 3, 5, and 8.

No single chunk size serves both. You either **miss the forest for the trees**, or **miss the trees for the forest**.

### Existing Workarounds and Their Limits

1. **Multi-vector retrieval** (e.g., parent-document retriever): Stores summaries alongside chunks, but the summaries are flat — they don't form a hierarchy.  
2. **Hypothetical Document Embeddings (HyDE)**: Rewrites the query, but doesn't change the index structure.  
3. **Map-reduce summarization**: Creates one summary per document, losing mid-level thematic groupings.  

What we *really* want is an index that **simultaneously represents information at every granularity**: precise details, paragraph-level context, section-level themes, chapter-level narratives, and whole-corpus overviews — all searchable in one pass.

This is exactly what **RAPTOR** provides.

## 2 · What Is RAPTOR?

RAPTOR — *Recursive Abstractive Processing for Tree-Organized Retrieval* — was introduced by Sarthi et al. at ICLR 2024. The key insight is deceptively simple:

> **Build a tree of summaries from bottom up, then index every node — leaves *and* internal nodes — in a single vector store.**

### The Tree Structure

```
Level 3 (Root):    [Global Summary of Entire Corpus]
                        /                    \
Level 2:      [Theme A Summary]        [Theme B Summary]
               /        \                /        \
Level 1:  [Cluster 0]  [Cluster 1]  [Cluster 2]  [Cluster 3]
           Summary       Summary      Summary      Summary
            / | \         / | \        / | \        / | \
Level 0:  c0 c1 c2     c3 c4 c5    c6 c7 c8    c9 c10 c11
          (original text chunks — the leaves)
```

- **Level 0**: Original text chunks (the leaves of the tree)
- **Level 1**: Cluster semantically similar chunks → summarize each cluster with an LLM
- **Level 2**: Cluster the Level 1 summaries → summarize again
- **Level 3+**: Continue until you reach a single root summary (or hit a max-depth limit)

### Why This Works

1. **Detail queries** naturally match Level 0 leaves (high cosine similarity with specific facts).
2. **Broad queries** naturally match higher-level summaries (which embed thematic overviews).
3. **Mid-level queries** match mid-tree nodes — something no flat index can provide.

The tree is built **offline** (expensive — many LLM summarization calls), but retrieval is **fast** (just a FAISS lookup across all levels).

## 3 · The RAPTOR Algorithm — Step by Step

### Offline: Tree Construction

```
Input:  List of text chunks C = [c₁, c₂, ..., cₙ]
Output: Tree T with nodes at levels 0, 1, ..., L

1. Set level ← 0, current_nodes ← C
2. WHILE |current_nodes| > 1 AND level < max_level:
   a. Embed all current_nodes → E = embed(current_nodes)
   b. Cluster E into k groups using KMeans or GMM
      - k = max(2, ⌈|current_nodes| / target_cluster_size⌉)
   c. For each cluster i = 1..k:
      - Collect texts in cluster i
      - Generate summary sᵢ = LLM_summarize(cluster_texts)
      - Record parent-child links: sᵢ → {texts in cluster i}
   d. Store all current_nodes in tree at `level`
   e. Set current_nodes ← [s₁, s₂, ..., sₖ]
   f. level ← level + 1
3. Store final node(s) at `level`
4. Return T
```

### Online: Retrieval (Two Strategies)

**Strategy A — Tree Traversal Retrieval**:  
Start at the root, find most relevant child at each level, drill down to leaves. This is top-down and precise, but requires level metadata.

**Strategy B — Collapsed Tree Retrieval** (simpler, often better):  
Flatten all nodes (every level) into a single FAISS index. Retrieve top-K regardless of level. The vector similarity naturally selects the right granularity.

### Key Design Choices

| Choice | Options | Trade-off |
|---|---|---|
| Clustering method | KMeans vs GMM | KMeans: hard assignment, simpler. GMM: soft, allows overlap |
| Number of clusters | Fixed k vs adaptive | Adaptive is better but needs heuristics |
| Summary length | Short (1-2 sentences) vs long | Longer preserves more info, but costs more tokens |
| Max tree depth | 2-4 levels typical | More levels = more LLM calls, diminishing returns past 3-4 |

## 4 · Environment Setup

We install everything in one shot. Key packages:
- **transformers + bitsandbytes**: Load Qwen3-14B in 4-bit NF4 quantization
- **sentence-transformers**: BGE embeddings
- **faiss-cpu**: Vector similarity search
- **scikit-learn**: KMeans and Gaussian Mixture clustering

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy scikit-learn

### Imports

In [ ]:
import numpy as np
import faiss
import torch
import time
import textwrap
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"FAISS version   : {faiss.__version__}")

## 5 · Load the Embedding Model

`BAAI/bge-base-en-v1.5` is a 110M parameter bi-encoder that maps text to 768-dimensional vectors. It is fast, accurate, and perfect for our clustering and retrieval steps.

> **Why BGE?** It consistently ranks near the top of the MTEB leaderboard for retrieval tasks, while being small enough to run alongside a 14B LLM on a single GPU.

In [ ]:
EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"
from google.colab import drive
drive.mount("/content/drive")

CACHE_DIR      = "/content/drive/MyDrive/models"

embed_model = SentenceTransformer(EMBED_MODEL_ID, cache_folder=CACHE_DIR)
print(f"Embedding model loaded: {EMBED_MODEL_ID}")
print(f"Embedding dimension  : {embed_model.get_sentence_embedding_dimension()}")

# Quick sanity check
test_emb = embed_model.encode(["Hello world"])
print(f"Test embedding shape : {test_emb.shape}")
print(f"Test embedding norm  : {np.linalg.norm(test_emb[0]):.4f}")

## 6 · Load the LLM (Qwen3-14B, 4-bit NF4)

We load `Qwen/Qwen3-14B` in 4-bit NF4 quantization using bitsandbytes. This reduces VRAM from ~28 GB (fp16) to ~8 GB, making it feasible on a single T4/L4 GPU.

The LLM serves one purpose in RAPTOR: **summarizing clusters of text**. It will be called many times during tree construction (one call per cluster per level), so we keep the generation concise.

In [ ]:
LLM_MODEL_ID = "Qwen/Qwen3-14B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_ID, cache_dir=CACHE_DIR, trust_remote_code=True
)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=CACHE_DIR,
    trust_remote_code=True,
)

print(f"LLM loaded: {LLM_MODEL_ID}")
print(f"Model dtype: {next(llm_model.parameters()).dtype}")
print(f"Device map : {llm_model.hf_device_map}")

### LLM Generation Helper

A thin wrapper around the model that handles chat-template formatting and generation parameters. We use `do_sample=False` for deterministic summaries.

In [ ]:
def generate(prompt: str, max_new_tokens: int = 300, temperature: float = 0.6) -> str:
    """Generate text from the LLM using chat template."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Respond concisely."},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
        )
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Quick sanity check
test_response = generate("In one sentence, what is machine learning?")
print(f"LLM test response: {test_response}")

## 7 · Synthetic Knowledge Base

We create a realistic multi-topic knowledge base about **Renewable Energy Technologies**. The 10 paragraphs span several subtopics — solar, wind, storage, grid integration, economics, policy, and environmental impact. This breadth ensures our RAPTOR tree can form meaningful clusters at multiple levels.

> In production, these would be chunks from document parsing. Here we use synthetic data so the notebook runs fully self-contained.

In [ ]:
KNOWLEDGE_BASE = [
    # --- Solar Energy (3 paragraphs) ---
    (
        "Solar photovoltaic (PV) technology converts sunlight directly into electricity using "
        "semiconductor materials, most commonly crystalline silicon. Modern monocrystalline "
        "panels achieve 22-24% efficiency in commercial production, while laboratory cells "
        "have surpassed 29%. The levelized cost of solar PV has fallen 90% since 2010, from "
        "$0.36/kWh to under $0.04/kWh in optimal locations. This dramatic cost reduction is "
        "driven by manufacturing scale in China, improved cell architectures like PERC and "
        "TOPCon, and thinner wafer technology. Solar PV now accounts for over 4% of global "
        "electricity generation and is the fastest-growing energy source worldwide."
    ),
    (
        "Concentrated solar power (CSP) uses mirrors or lenses to focus sunlight onto a "
        "receiver, heating a working fluid to drive a turbine. Unlike PV, CSP can incorporate "
        "thermal energy storage using molten salt, enabling electricity generation for 6-15 "
        "hours after sunset. The Noor-Ouarzazate complex in Morocco, the world's largest CSP "
        "plant at 580 MW, provides reliable power to over 2 million homes. However, CSP "
        "requires direct normal irradiance (DNI) above 2000 kWh/m²/year, limiting deployment "
        "to arid regions. CSP costs remain higher than PV at $0.08-0.12/kWh, but the built-in "
        "storage capability makes it valuable for grid stability in sun-rich countries."
    ),
    (
        "Perovskite solar cells represent the most promising emerging PV technology. These "
        "thin-film cells use metal halide perovskite absorbers that can be deposited from "
        "solution at low temperatures, potentially enabling roll-to-roll manufacturing on "
        "flexible substrates. Single-junction perovskites have reached 26.1% efficiency, and "
        "perovskite-silicon tandems have achieved 33.9% — exceeding the theoretical single-"
        "junction silicon limit. The major challenge is long-term stability: perovskites "
        "degrade when exposed to moisture, oxygen, and UV light. Current research focuses on "
        "encapsulation techniques and compositional engineering to achieve 25-year lifetimes "
        "comparable to silicon panels."
    ),
    # --- Wind Energy (2 paragraphs) ---
    (
        "Onshore wind turbines have grown dramatically in scale over the past two decades. "
        "Modern turbines feature rotor diameters exceeding 170 meters and hub heights above "
        "120 meters, with individual capacities of 6-7 MW. Larger rotors capture more energy "
        "from lower wind speeds, improving capacity factors from 25% to over 40% in good "
        "sites. The global onshore wind fleet exceeded 900 GW by 2024. Key challenges include "
        "land use conflicts, noise regulations limiting tip-speed, avian mortality, and the "
        "need for grid reinforcement in remote windy areas. Despite these challenges, onshore "
        "wind remains the cheapest source of new electricity in many regions at $0.03-0.05/kWh."
    ),
    (
        "Offshore wind is experiencing explosive growth, particularly in Europe and Asia. "
        "Turbines rated at 14-15 MW are now being deployed, with 20 MW designs in development. "
        "Fixed-bottom foundations dominate in waters up to 60 meters deep, while floating "
        "platforms are opening vast new areas in deeper waters — potentially 4x the resource "
        "of fixed-bottom sites. The UK leads with 14 GW installed, followed by China at 12 GW. "
        "Offshore wind costs have fallen to $0.05-0.08/kWh thanks to larger turbines, "
        "optimized logistics, and competitive auction mechanisms. The higher capacity factors "
        "(45-55%) and proximity to coastal demand centers make offshore wind a cornerstone of "
        "many national decarbonization strategies."
    ),
    # --- Energy Storage (2 paragraphs) ---
    (
        "Lithium-ion batteries dominate the energy storage market with over 90% market share. "
        "Battery pack costs have fallen from $1,200/kWh in 2010 to approximately $140/kWh in "
        "2024, with projections reaching $80/kWh by 2030. Lithium iron phosphate (LFP) "
        "chemistry has gained significant market share due to its superior safety, longer cycle "
        "life (4000+ cycles), and avoidance of cobalt and nickel. Grid-scale battery storage "
        "installations reached 45 GWh globally in 2023. Four-hour duration systems are now "
        "cost-competitive with natural gas peaker plants in many markets. However, lithium-ion "
        "is poorly suited for long-duration storage (beyond 8 hours) due to the linear scaling "
        "of cost with duration."
    ),
    (
        "Long-duration energy storage (LDES) technologies are essential for achieving grids "
        "with over 80% renewable penetration. Promising approaches include iron-air batteries, "
        "which use abundant iron and can store energy for 100+ hours at very low cost per kWh; "
        "compressed air energy storage (CAES), which stores energy in underground caverns; and "
        "green hydrogen produced via electrolysis, which can be stored in salt caverns for "
        "seasonal shifting. Flow batteries using vanadium or zinc-bromine electrolytes offer "
        "10-24 hour durations with decoupled power and energy capacity. The US Department of "
        "Energy's Long Duration Storage Shot aims to reduce LDES costs to $0.05/kWh by 2030, "
        "which would make 100% renewable grids economically viable."
    ),
    # --- Grid Integration & Policy (2 paragraphs) ---
    (
        "Integrating high shares of variable renewable energy into electricity grids requires "
        "fundamental changes to grid architecture and market design. Key strategies include: "
        "enhanced interconnection between regions to smooth variability geographically; demand "
        "response programs that shift flexible loads (EV charging, water heating, industrial "
        "processes) to periods of high renewable output; advanced weather forecasting to "
        "improve scheduling accuracy; and grid-forming inverters that provide synthetic "
        "inertia to maintain frequency stability without synchronous generators. Countries "
        "like Denmark (80% wind+solar) and South Australia (70% wind+solar) demonstrate that "
        "very high renewable shares are technically achievable with proper grid management."
    ),
    (
        "Government policy plays a decisive role in renewable energy deployment speed. Feed-in "
        "tariffs drove early solar and wind adoption in Germany and Spain. Competitive auction "
        "mechanisms now dominate, driving costs down through market competition. Carbon pricing "
        "through emissions trading systems (EU ETS, $50-100/tCO₂) or carbon taxes (Canada, "
        "$65/tCO₂) creates economic incentives to switch from fossil fuels. The US Inflation "
        "Reduction Act (IRA) of 2022 allocated $369 billion in clean energy subsidies, "
        "triggering over $200 billion in private investment within 18 months. China's "
        "industrial policy has made it the dominant manufacturer of solar panels, batteries, "
        "and EVs, controlling 80%+ of the solar PV supply chain."
    ),
    # --- Environmental Impact (1 paragraph) ---
    (
        "While renewable energy dramatically reduces greenhouse gas emissions compared to "
        "fossil fuels (solar PV: 20-50 gCO₂/kWh vs coal: 820-1200 gCO₂/kWh lifecycle), it "
        "is not without environmental concerns. Solar farms require 5-10 acres per MW, "
        "creating land-use competition with agriculture and ecosystems. Wind turbines cause "
        "an estimated 140,000-500,000 bird deaths annually in the US, though this is far fewer "
        "than buildings (600 million) or cats (2.4 billion). Battery manufacturing requires "
        "mining of lithium, cobalt, and nickel, with associated water use and habitat "
        "disruption. End-of-life recycling infrastructure for solar panels and turbine blades "
        "remains underdeveloped. A holistic assessment shows renewables have 10-20x lower "
        "lifecycle environmental impact than fossil fuels across most metrics, but responsible "
        "deployment requires careful siting, biodiversity offsets, and circular economy design."
    ),
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} paragraphs")
for i, para in enumerate(KNOWLEDGE_BASE):
    words = len(para.split())
    print(f"  Chunk {i}: {words:3d} words — {para[:70]}...")

## 8 · Step 1 — Embed All Chunks (Level 0)

The first step of RAPTOR is to embed every original chunk. These embeddings serve two purposes:
1. **Clustering**: Group semantically similar chunks together
2. **Retrieval**: Include Level 0 nodes in the final FAISS index

BGE models expect a prefix `"Represent this sentence: "` for document encoding, but sentence-transformers handles this automatically when the model config includes the prompt.

In [ ]:
# Embed the original chunks
chunk_embeddings = embed_model.encode(KNOWLEDGE_BASE, show_progress_bar=True, normalize_embeddings=True)
chunk_embeddings = np.array(chunk_embeddings, dtype=np.float32)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")
print(f"Embedding dtype       : {chunk_embeddings.dtype}")

# Show pairwise similarity to confirm semantic structure
sim_matrix = chunk_embeddings @ chunk_embeddings.T
print("\nPairwise cosine similarity (first 5 chunks):")
labels = ["Solar-PV", "Solar-CSP", "Perovskite", "Wind-On", "Wind-Off"]
print(f"{'':>12}", end="")
for lbl in labels:
    print(f"{lbl:>12}", end="")
print()
for i in range(5):
    print(f"{labels[i]:>12}", end="")
    for j in range(5):
        print(f"{sim_matrix[i,j]:12.3f}", end="")
    print()

## 9 · Tree Data Structure

Before building the tree, we define a clean data structure to hold each node. Every node stores:
- Its text content
- Its embedding vector
- Its level in the tree (0 = original chunk)
- Its cluster assignment
- A list of child node indices (empty for leaves)

In [ ]:
@dataclass
class TreeNode:
    """A node in the RAPTOR tree."""
    text: str
    embedding: np.ndarray
    level: int
    cluster_id: int = -1
    children: List[int] = field(default_factory=list)  # indices into global node list
    node_id: int = -1

    def __repr__(self):
        return (f"TreeNode(id={self.node_id}, level={self.level}, "
                f"cluster={self.cluster_id}, children={self.children}, "
                f"text='{self.text[:60]}...')")

# Create Level 0 nodes from our knowledge base
all_nodes: List[TreeNode] = []
for i, (text, emb) in enumerate(zip(KNOWLEDGE_BASE, chunk_embeddings)):
    node = TreeNode(text=text, embedding=emb, level=0, node_id=i)
    all_nodes.append(node)

print(f"Created {len(all_nodes)} leaf nodes (Level 0)")
print(f"Example: {all_nodes[0]}")

## 10 · Step 2 — Cluster Similar Chunks

We cluster the Level 0 embeddings so that semantically related chunks are grouped together. Each cluster will be summarized into a single Level 1 node.

### Choosing k (number of clusters)

The original RAPTOR paper uses GMM with soft assignments. For clarity, we start with **KMeans** (hard assignments) and show GMM as an alternative. The heuristic for k:

```
k = max(2, ceil(n / target_cluster_size))
```

With 10 chunks and a target of 3 chunks per cluster, we get k = 4. This creates 4 Level 1 summaries.

In [ ]:
import math

def compute_num_clusters(n_items: int, target_size: int = 3, min_k: int = 2) -> int:
    """Compute number of clusters using a simple heuristic."""
    k = max(min_k, math.ceil(n_items / target_size))
    return min(k, n_items)  # can't have more clusters than items

def cluster_embeddings(embeddings: np.ndarray, n_clusters: int,
                       method: str = "kmeans") -> np.ndarray:
    """Cluster embeddings using KMeans or GMM. Returns cluster labels."""
    if method == "kmeans":
        model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = model.fit_predict(embeddings)
    elif method == "gmm":
        model = GaussianMixture(n_components=n_clusters, random_state=42,
                                covariance_type="full")
        labels = model.fit_predict(embeddings)
    else:
        raise ValueError(f"Unknown method: {method}")
    return labels

# Cluster Level 0
n_clusters_l0 = compute_num_clusters(len(KNOWLEDGE_BASE))
labels_l0 = cluster_embeddings(chunk_embeddings, n_clusters_l0, method="kmeans")

print(f"Number of clusters: {n_clusters_l0}")
print(f"Cluster assignments: {labels_l0}")
print()
for c in range(n_clusters_l0):
    members = [i for i, l in enumerate(labels_l0) if l == c]
    print(f"Cluster {c}: chunks {members}")
    for m in members:
        print(f"    → {KNOWLEDGE_BASE[m][:80]}...")

## 11 · Step 3 — Summarize Each Cluster with the LLM

For each cluster, we concatenate all member texts and ask the LLM to produce a concise summary. This is the most expensive step — one LLM call per cluster, per level.

The summary should:
- Capture the **key themes** shared by the cluster members
- Preserve **specific facts and numbers** where possible
- Be concise (2-4 sentences) to keep higher-level nodes information-dense

In [ ]:
def summarize_cluster(texts: List[str]) -> str:
    """Summarize a cluster of texts using the LLM."""
    combined = "\n\n".join(f"[Passage {i+1}]: {t}" for i, t in enumerate(texts))
    prompt = (
        "Below are several related text passages. Write a concise summary (2-4 sentences) "
        "that captures the key themes and important facts across all passages.\n\n"
        f"{combined}\n\n"
        "Summary:"
    )
    return generate(prompt, max_new_tokens=200)

# Summarize each Level 0 cluster
level1_summaries = []
level1_children  = []

for c in range(n_clusters_l0):
    member_indices = [i for i, l in enumerate(labels_l0) if l == c]
    member_texts = [KNOWLEDGE_BASE[i] for i in member_indices]
    print(f"\n{'='*60}")
    print(f"Cluster {c}: summarizing {len(member_texts)} chunks (indices {member_indices})")
    summary = summarize_cluster(member_texts)
    level1_summaries.append(summary)
    level1_children.append(member_indices)
    print(f"Summary: {summary[:200]}..." if len(summary) > 200 else f"Summary: {summary}")

print(f"\n\nGenerated {len(level1_summaries)} Level 1 summaries.")

## 12 · Step 4 — Build the Full RAPTOR Tree Recursively

Now we put it all together. The `build_raptor_tree` function:
1. Takes the Level 0 nodes as input
2. At each level: embed → cluster → summarize → create new nodes
3. Stops when only 1 node remains (the root) or max depth is reached
4. Returns the complete list of all nodes across all levels

This is the **core algorithm** of RAPTOR — everything else is retrieval on top of this tree.

In [ ]:
def build_raptor_tree(chunks: List[str], embeddings_l0: np.ndarray,
                      max_levels: int = 3, target_cluster_size: int = 3,
                      cluster_method: str = "kmeans") -> List[TreeNode]:
    """
    Build the complete RAPTOR tree from leaf chunks.
    Returns list of ALL nodes (leaves + internal + root).
    """
    all_nodes = []

    # --- Level 0: Original chunks ---
    for i, (text, emb) in enumerate(zip(chunks, embeddings_l0)):
        all_nodes.append(TreeNode(
            text=text, embedding=emb, level=0, node_id=i
        ))
    print(f"Level 0: {len(chunks)} leaf nodes")

    current_level_indices = list(range(len(chunks)))

    for level in range(1, max_levels + 1):
        n_current = len(current_level_indices)
        if n_current <= 1:
            print(f"  Stopping: only {n_current} node(s) at previous level.")
            break

        # Get embeddings of current level nodes
        current_embeddings = np.array(
            [all_nodes[idx].embedding for idx in current_level_indices]
        )

        # Cluster
        k = compute_num_clusters(n_current, target_cluster_size)
        if k >= n_current:
            k = max(1, n_current // 2)
        labels = cluster_embeddings(current_embeddings, k, method=cluster_method)

        print(f"Level {level}: clustering {n_current} nodes into {k} clusters")

        new_level_indices = []
        for c in range(k):
            member_positions = [i for i, l in enumerate(labels) if l == c]
            member_node_ids = [current_level_indices[p] for p in member_positions]
            member_texts = [all_nodes[idx].text for idx in member_node_ids]

            # Summarize
            summary = summarize_cluster(member_texts)

            # Embed the summary
            summary_emb = embed_model.encode([summary], normalize_embeddings=True)[0]

            # Create new node
            new_id = len(all_nodes)
            node = TreeNode(
                text=summary,
                embedding=summary_emb.astype(np.float32),
                level=level,
                cluster_id=c,
                children=member_node_ids,
                node_id=new_id,
            )
            all_nodes.append(node)
            new_level_indices.append(new_id)

            print(f"  Cluster {c}: {len(member_node_ids)} children → node {new_id}")
            print(f"    Summary: {summary[:100]}...")

        current_level_indices = new_level_indices

    print(f"\nTree complete: {len(all_nodes)} total nodes")
    return all_nodes

# Build the tree!
tree_nodes = build_raptor_tree(
    KNOWLEDGE_BASE, chunk_embeddings,
    max_levels=3, target_cluster_size=3, cluster_method="kmeans"
)

## 13 · Visualize the RAPTOR Tree

Let's inspect the tree structure: how many nodes at each level, the parent-child relationships, and summary snippets. This visualization helps build intuition for how RAPTOR organizes information hierarchically.

In [ ]:
def visualize_tree(nodes: List[TreeNode]):
    """Print a structured view of the RAPTOR tree."""
    max_level = max(n.level for n in nodes)
    print(f"RAPTOR Tree — {len(nodes)} nodes, {max_level + 1} levels")
    print("=" * 70)

    for level in range(max_level, -1, -1):
        level_nodes = [n for n in nodes if n.level == level]
        label = "ROOT" if level == max_level else f"Level {level}"
        if level == 0:
            label = "LEAVES (original chunks)"
        print(f"\n{'─'*70}")
        print(f"  {label} — {len(level_nodes)} node(s)")
        print(f"{'─'*70}")
        for n in level_nodes:
            children_str = f", children={n.children}" if n.children else ""
            snippet = n.text[:90].replace('\n', ' ')
            print(f"  [Node {n.node_id:2d}] (cluster {n.cluster_id}){children_str}")
            print(f"           \"{snippet}...\"")

visualize_tree(tree_nodes)

## 14 · Build the Collapsed FAISS Index

In **collapsed tree retrieval**, we flatten all nodes — leaves, intermediate summaries, and root — into a single FAISS index. When a query arrives, we search across all levels simultaneously. The vector similarity naturally selects the right granularity:

- A detail question will match a Level 0 leaf
- A thematic question will match a Level 1-2 summary
- A "what's this corpus about?" question will match the root

This is simpler and often more effective than tree traversal.

In [ ]:
def build_collapsed_index(nodes: List[TreeNode]) -> faiss.IndexFlatIP:
    """
    Build a flat FAISS inner-product index from all tree nodes.
    (We use inner product because embeddings are L2-normalized → IP = cosine similarity.)
    """
    dim = nodes[0].embedding.shape[0]
    index = faiss.IndexFlatIP(dim)
    embeddings = np.array([n.embedding for n in nodes], dtype=np.float32)
    index.add(embeddings)
    return index

collapsed_index = build_collapsed_index(tree_nodes)
print(f"FAISS index built: {collapsed_index.ntotal} vectors, dimension {collapsed_index.d}")
print(f"  Level 0 (leaves)    : {sum(1 for n in tree_nodes if n.level == 0)} nodes")
print(f"  Level 1 (summaries) : {sum(1 for n in tree_nodes if n.level == 1)} nodes")
print(f"  Level 2+ (higher)   : {sum(1 for n in tree_nodes if n.level >= 2)} nodes")

## 15 · Collapsed Tree Retrieval

The collapsed retrieval function is beautifully simple: embed the query, search the index, return the top-K results with their level metadata. The key insight is that **the FAISS index doesn't know or care about tree levels** — it just returns the most similar vectors.

In [ ]:
def collapsed_tree_retrieve(query: str, index: faiss.IndexFlatIP,
                            nodes: List[TreeNode], k: int = 5
                            ) -> List[Tuple[TreeNode, float]]:
    """
    Collapsed tree retrieval: search ALL levels at once, return top-K.
    Returns list of (node, similarity_score) tuples.
    """
    query_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(query_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(nodes):
            results.append((nodes[idx], float(score)))
    return results

def print_retrieval_results(results: List[Tuple[TreeNode, float]], query: str):
    """Pretty-print retrieval results with level info."""
    print(f"Query: \"{query}\"")
    print(f"{'─'*70}")
    for rank, (node, score) in enumerate(results, 1):
        level_label = {0: "LEAF", 1: "L1-Summary", 2: "L2-Summary", 3: "ROOT"}
        label = level_label.get(node.level, f"L{node.level}")
        snippet = node.text[:120].replace('\n', ' ')
        print(f"  #{rank} [{label:>11}] (sim={score:.4f}, node={node.node_id})")
        print(f"      \"{snippet}...\"")
    print()

# Test with a DETAIL query — should hit leaves
detail_query = "What is the efficiency of perovskite solar cells?"
results = collapsed_tree_retrieve(detail_query, collapsed_index, tree_nodes, k=5)
print_retrieval_results(results, detail_query)

### Broad Query — Should Hit Higher Levels

Now let's try a thematic query that requires synthesizing across multiple topics. This should naturally match higher-level summaries.

In [ ]:
broad_query = "What are the main renewable energy technologies and their current status?"
results = collapsed_tree_retrieve(broad_query, collapsed_index, tree_nodes, k=5)
print_retrieval_results(results, broad_query)

## 16 · Tree Traversal Retrieval

An alternative to collapsed retrieval is **tree traversal**: start at the root, find the most relevant children, then drill down level by level. This is more structured but requires more queries to the index.

The algorithm:
1. Start at the highest level in the tree
2. Retrieve top-k nodes at that level
3. For each retrieved node, retrieve its children at the level below
4. Repeat until reaching Level 0 (leaves)
5. Return all collected nodes across all levels

In [ ]:
def tree_traversal_retrieve(query: str, nodes: List[TreeNode],
                            top_k_per_level: int = 2) -> List[Tuple[TreeNode, float]]:
    """
    Tree traversal retrieval: top-down from root to leaves.
    At each level, pick the best nodes and drill into their children.
    """
    query_emb = embed_model.encode([query], normalize_embeddings=True)[0]
    max_level = max(n.level for n in nodes)
    collected = []

    # Start with all nodes at the highest level
    candidate_ids = [n.node_id for n in nodes if n.level == max_level]

    for level in range(max_level, -1, -1):
        if not candidate_ids:
            break

        # Score candidates at this level
        candidates = [nodes[nid] for nid in candidate_ids]
        cand_embs = np.array([n.embedding for n in candidates], dtype=np.float32)
        scores = cand_embs @ query_emb  # cosine similarity (normalized)

        # Select top-k
        k = min(top_k_per_level, len(candidates))
        top_indices = np.argsort(scores)[::-1][:k]

        # Collect selected nodes
        next_candidate_ids = []
        for idx in top_indices:
            node = candidates[idx]
            collected.append((node, float(scores[idx])))
            # Queue their children for the next level
            next_candidate_ids.extend(node.children)

        candidate_ids = next_candidate_ids

    return collected

# Test tree traversal with a detail query
detail_query = "How much does lithium-ion battery storage cost per kWh?"
traversal_results = tree_traversal_retrieve(detail_query, tree_nodes, top_k_per_level=2)

print(f"Query: \"{detail_query}\"")
print(f"Tree traversal returned {len(traversal_results)} nodes:\n")
for rank, (node, score) in enumerate(traversal_results, 1):
    level_label = {0: "LEAF", 1: "L1-Summary", 2: "L2-Summary", 3: "ROOT"}
    label = level_label.get(node.level, f"L{node.level}")
    print(f"  #{rank} [{label:>11}] (sim={score:.4f}, node={node.node_id})")
    print(f"      \"{node.text[:120]}...\"")

## 17 · Complete RAG Pipeline with RAPTOR

Now we wire everything into an end-to-end RAG pipeline:
1. **Retrieve** relevant nodes using collapsed tree retrieval
2. **Build context** from retrieved nodes (sorted by level for coherence)
3. **Generate answer** using the LLM with the retrieved context

The key difference from flat RAG is that our context may include both specific paragraphs (leaves) and broader summaries (internal nodes), giving the LLM both detail and big-picture information.

In [ ]:
def raptor_rag(query: str, index: faiss.IndexFlatIP,
               nodes: List[TreeNode], k: int = 5) -> Dict:
    """
    End-to-end RAG with RAPTOR collapsed tree retrieval.
    Returns dict with query, retrieved nodes, context, and answer.
    """
    # Retrieve
    retrieved = collapsed_tree_retrieve(query, index, nodes, k=k)

    # Sort by level (high→low) so broad context comes before details
    retrieved_sorted = sorted(retrieved, key=lambda x: -x[0].level)

    # Build context
    context_parts = []
    for node, score in retrieved_sorted:
        level_label = f"[Level {node.level}]"
        context_parts.append(f"{level_label} {node.text}")
    context = "\n\n".join(context_parts)

    # Generate answer
    prompt = (
        f"Based on the following context, answer the question accurately and concisely.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )
    answer = generate(prompt, max_new_tokens=300)

    return {
        "query": query,
        "retrieved": retrieved_sorted,
        "context": context,
        "answer": answer,
    }

# Run a complete RAG query
result = raptor_rag(
    "What are the main challenges with perovskite solar cells and how close are they to commercialization?",
    collapsed_index, tree_nodes, k=4
)

print(f"Query: {result['query']}\n")
print("Retrieved nodes:")
for node, score in result['retrieved']:
    print(f"  Level {node.level} (sim={score:.4f}): {node.text[:80]}...")
print(f"\nAnswer:\n{result['answer']}")

### RAG with a Broad Query

Let's test with a deliberately broad question to see if higher-level summaries contribute to a better answer.

In [ ]:
result2 = raptor_rag(
    "Give me a comprehensive overview of the renewable energy landscape including generation, storage, and policy.",
    collapsed_index, tree_nodes, k=5
)

print(f"Query: {result2['query']}\n")
print("Retrieved nodes (level distribution):")
from collections import Counter
level_counts = Counter(node.level for node, _ in result2['retrieved'])
for level in sorted(level_counts):
    print(f"  Level {level}: {level_counts[level]} node(s)")
print(f"\nAnswer:\n{result2['answer']}")

## 18 · Comparison: Flat Retrieval vs RAPTOR

To demonstrate the value of RAPTOR, we build a **flat index** (Level 0 only) and compare retrieval quality against the collapsed RAPTOR tree index.

We test two query types:
1. **Detail query**: Expects specific factual information from a single chunk
2. **Broad query**: Expects thematic synthesis across multiple topics

The hypothesis: flat retrieval matches RAPTOR on detail queries, but **RAPTOR significantly outperforms on broad queries** because higher-level summaries provide pre-synthesized thematic content.

In [ ]:
# Build flat index (Level 0 only)
leaf_nodes = [n for n in tree_nodes if n.level == 0]
flat_index = faiss.IndexFlatIP(leaf_nodes[0].embedding.shape[0])
flat_embeddings = np.array([n.embedding for n in leaf_nodes], dtype=np.float32)
flat_index.add(flat_embeddings)
print(f"Flat index: {flat_index.ntotal} vectors (Level 0 only)")
print(f"RAPTOR index: {collapsed_index.ntotal} vectors (all levels)\n")

# Test queries
test_queries = [
    ("detail", "What is the cost per kWh of offshore wind energy?"),
    ("detail", "What efficiency have perovskite-silicon tandem cells achieved?"),
    ("broad",  "How do renewable energy technologies compare in terms of cost, scalability, and environmental impact?"),
    ("broad",  "What role does government policy play in accelerating the energy transition?"),
]

for qtype, query in test_queries:
    print(f"\n{'='*70}")
    print(f"[{qtype.upper()}] {query}")
    print(f"{'='*70}")

    # Flat retrieval
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    flat_scores, flat_idx = flat_index.search(q_emb, 3)
    print(f"\n  FLAT (top-3):")
    for rank, (score, idx) in enumerate(zip(flat_scores[0], flat_idx[0]), 1):
        print(f"    #{rank} sim={score:.4f} → {leaf_nodes[idx].text[:90]}...")

    # RAPTOR retrieval
    raptor_results = collapsed_tree_retrieve(query, collapsed_index, tree_nodes, k=3)
    print(f"\n  RAPTOR (top-3):")
    for rank, (node, score) in enumerate(raptor_results, 1):
        lbl = f"L{node.level}"
        print(f"    #{rank} sim={score:.4f} [{lbl}] → {node.text[:90]}...")

## 19 · Alternative: Gaussian Mixture Model Clustering

The original RAPTOR paper uses **Gaussian Mixture Models** (GMM) instead of KMeans. The key advantage: GMM provides **soft cluster assignments** — a chunk can partially belong to multiple clusters, which better captures the reality that a paragraph about "solar panel manufacturing costs" relates to both "solar technology" and "economics".

Let's rebuild with GMM and compare the cluster assignments.

In [ ]:
# Cluster with GMM instead of KMeans
labels_gmm = cluster_embeddings(chunk_embeddings, n_clusters_l0, method="gmm")

print("Comparison of cluster assignments:")
print(f"{'Chunk':>8} {'KMeans':>8} {'GMM':>8}  Topic")
print("-" * 60)
topics = ["Solar-PV", "Solar-CSP", "Perovskite", "Wind-Onshore", "Wind-Offshore",
          "Battery-LiIon", "Storage-LDES", "Grid-Integration", "Policy", "Environment"]
for i in range(len(KNOWLEDGE_BASE)):
    match = "✓" if labels_l0[i] == labels_gmm[i] else "✗"
    print(f"{i:>8} {labels_l0[i]:>8} {labels_gmm[i]:>8}  {match} {topics[i]}")

agreement = sum(1 for a, b in zip(labels_l0, labels_gmm) if a == b) / len(labels_l0)
print(f"\nAgreement: {agreement:.0%}")

# Show GMM soft assignments (probabilities)
gmm = GaussianMixture(n_components=n_clusters_l0, random_state=42, covariance_type="full")
gmm.fit(chunk_embeddings)
probs = gmm.predict_proba(chunk_embeddings)
print(f"\nGMM soft assignments (probability per cluster):")
print(f"{'Chunk':>8}", end="")
for c in range(n_clusters_l0):
    print(f"{'C'+str(c):>8}", end="")
print()
for i in range(len(KNOWLEDGE_BASE)):
    print(f"{i:>8}", end="")
    for c in range(n_clusters_l0):
        val = probs[i, c]
        marker = f"{val:.3f}" if val > 0.01 else "  -   "
        print(f"{marker:>8}", end="")
    print(f"  {topics[i]}")

## 20 · Analysis: Tree Statistics and Compression Ratio

Let's examine how much the RAPTOR tree compresses information at each level, and estimate the computational cost of tree construction.

In [ ]:
def analyze_tree(nodes: List[TreeNode]):
    """Print statistics about the RAPTOR tree."""
    max_level = max(n.level for n in nodes)
    total_chars = 0

    print(f"{'Level':>6} {'Nodes':>6} {'Avg Chars':>10} {'Total Chars':>12} {'LLM Calls':>10}")
    print("─" * 52)

    for level in range(max_level + 1):
        level_nodes = [n for n in nodes if n.level == level]
        chars = [len(n.text) for n in level_nodes]
        avg_chars = np.mean(chars)
        total = sum(chars)
        total_chars += total
        llm_calls = len(level_nodes) if level > 0 else 0  # no LLM calls for leaves
        print(f"{level:>6} {len(level_nodes):>6} {avg_chars:>10.0f} {total:>12,} {llm_calls:>10}")

    total_llm_calls = sum(1 for n in nodes if n.level > 0)
    leaf_chars = sum(len(n.text) for n in nodes if n.level == 0)
    summary_chars = sum(len(n.text) for n in nodes if n.level > 0)

    print("─" * 52)
    print(f"Total nodes       : {len(nodes)}")
    print(f"Total LLM calls   : {total_llm_calls}")
    print(f"Leaf text          : {leaf_chars:,} characters")
    print(f"Summary text       : {summary_chars:,} characters")
    print(f"Compression ratio  : {leaf_chars / max(summary_chars, 1):.1f}x (leaf / summary)")
    print(f"Index size increase: {len(nodes) / sum(1 for n in nodes if n.level == 0):.1f}x vs flat")

analyze_tree(tree_nodes)

## 21 · Synthesis: When to Use RAPTOR

### Construction Cost

RAPTOR is **expensive to build**. Each non-leaf node requires one LLM summarization call. For a corpus with *n* chunks and a branching factor *b*:

$$\text{LLM calls} \approx \frac{n}{b} + \frac{n}{b^2} + \cdots \approx \frac{n}{b-1}$$

For our 10-chunk example this was modest (~6 calls), but for 10,000 chunks with *b* = 5, it's ~2,500 LLM calls — significant cost and latency.

### When RAPTOR Shines ✅

| Scenario | Why RAPTOR Helps |
|---|---|
| **Long documents** (books, manuals, legal contracts) | Multi-level summaries capture both section and chapter themes |
| **Multi-scale questions** | "What's the main argument?" and "What's the footnote on page 47?" both work |
| **Corpus with clear thematic structure** | Clustering naturally finds meaningful groups |
| **Offline indexing is acceptable** | Tree can be built once and reused for many queries |

### When RAPTOR May Not Be Worth It ❌

| Scenario | Why |
|---|---|
| **Small documents** (<20 chunks) | Flat retrieval with good chunking often suffices |
| **Rapidly changing corpus** | Tree must be rebuilt when documents change |
| **Only detail questions** | If users only ask precise factual questions, flat retrieval is fine |
| **Tight LLM budget** | Construction cost may exceed the benefit |

### Alternatives

| Approach | Relationship to RAPTOR |
|---|---|
| **Parent-document retriever** | Simpler (no LLM calls), but only 2 levels (chunk + parent) |
| **Multi-vector retrieval** | Stores summaries as alternative representations, but no hierarchy |
| **Graph RAG** | Builds a knowledge graph instead of a summary tree — better for entity-centric questions |
| **Hierarchical indices** | Similar idea but typically uses metadata-based routing, not embedding similarity |

### Key Takeaway

> RAPTOR bridges the granularity gap by building a **searchable hierarchy of summaries**. It's most valuable for long, thematically rich documents where users ask questions at multiple levels of abstraction.

## 22 · Final End-to-End Demonstration

Let's run three different query types through the full RAPTOR RAG pipeline to see the system in action.

In [ ]:
demo_queries = [
    "What is the specific efficiency record for perovskite-silicon tandem solar cells?",
    "How do different energy storage technologies compare for grid-scale applications?",
    "Provide a high-level summary of the current state of renewable energy worldwide.",
]

for i, q in enumerate(demo_queries, 1):
    print(f"\n{'='*70}")
    print(f"Demo {i}: {q}")
    print(f"{'='*70}")
    result = raptor_rag(q, collapsed_index, tree_nodes, k=4)
    levels_hit = [n.level for n, _ in result['retrieved']]
    print(f"Levels hit: {levels_hit}")
    print(f"\nAnswer:\n{textwrap.fill(result['answer'], width=80)}\n")

## Summary & Key Takeaways

In this notebook we built **RAPTOR from scratch** — no frameworks, no abstractions.

### What We Implemented

1. **Embedding** with `BAAI/bge-base-en-v1.5` (sentence-transformers)
2. **Clustering** with scikit-learn KMeans and Gaussian Mixture Models
3. **Summarization** with `Qwen/Qwen3-14B` (4-bit NF4 quantization)
4. **Recursive tree construction** — embed → cluster → summarize → repeat
5. **Collapsed tree retrieval** — all levels in one FAISS index
6. **Tree traversal retrieval** — top-down from root to leaves
7. **Full RAG pipeline** — retrieve from RAPTOR tree → generate answer

### Core Insight

> The granularity problem in RAG isn't about finding the right chunk size — it's about **indexing information at every scale simultaneously**. RAPTOR achieves this by building a tree of summaries and searching all levels at once.

### Trade-offs

- **Build cost**: O(n/b) LLM calls for n chunks and branching factor b
- **Index overhead**: ~1.5-2x more vectors than flat index
- **Query speed**: Same as flat retrieval (single FAISS search)
- **Retrieval quality**: Significantly better for broad/thematic queries; comparable for detail queries

### Reference

Sarthi, P., Abdullah, S., Tuli, A., Khanna, S., Goldie, A., & Manning, C. D. (2024).  
*RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval.*  
ICLR 2024. [arXiv:2401.18059](https://arxiv.org/abs/2401.18059)